# NFL Officiating Patterns: Data Preparation and Cleaning

In [1]:
import sys
print(sys.version)
import numpy as np
print(np.__version__)
import pandas as pd
print(pd.__version__)
import matplotlib.pyplot as plt
import nflreadpy as nfl

3.12.7 | packaged by Anaconda, Inc. | (main, Oct  4 2024, 08:22:19) [Clang 14.0.6 ]
1.26.4
2.2.2


## Loading the Datasets with nflreadpy
### Play-By-Play Data (2015-2024)
We will start by loading play-by-play data from the 2015 season to the 2024 season using nflreadpy. This contains most of the play-specific information we will need for our analysis. We chose this time-frame in order to observe more accurate temporal patterns within the data. Additionally, the federal ban on sports betting was struck down in May of 2018 (with sports betting being legal in 38 states as of 2025), so this span will contain enough data points to observe officiating changes (if any) both before and after this occurred. 

Side-note: nflreadpy imports the data as a Polars dataframe, so we will also be converting all to Pandas before moving forward.

In [120]:
# Load season play-by-play data
pbp = nfl.load_pbp([2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024])

In [121]:
# Converting to pandas
pbp = pbp.to_pandas()

In [122]:
pbp.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 483605 entries, 0 to 483604
Columns: 372 entries, play_id to pass_oe
dtypes: float64(199), int32(8), object(165)
memory usage: 1.3+ GB


In [123]:
pbp.head()

,play_id,game_id,old_game_id,home_team,away_team,season_type,week,posteam,posteam_type,defteam,...,out_of_bounds,home_opening_kickoff,qb_epa,xyac_epa,xyac_mean_yardage,xyac_median_yardage,xyac_success,xyac_fd,xpass,pass_oe
0,1.0,2015_01_BAL_DEN,2015091309,DEN,BAL,REG,1,None,None,None,...,0.0,0.0,-0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,36.0,2015_01_BAL_DEN,2015091309,DEN,BAL,REG,1,BAL,away,DEN,...,0.0,0.0,-0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,51.0,2015_01_BAL_DEN,2015091309,DEN,BAL,REG,1,BAL,away,DEN,...,0.0,0.0,-0.337139,0.691718,4.699278,3.0,0.678964,0.225919,0.456481,54.351911
3,75.0,2015_01_BAL_DEN,2015091309,DEN,BAL,REG,1,BAL,away,DEN,...,0.0,0.0,-0.262481,NaN,NaN,NaN,NaN,NaN,0.545905,-54.590458
4,96.0,2015_01_BAL_DEN,2015091309,DEN,BAL,REG,1,BAL,away,DEN,...,0.0,0.0,1.661242,1.794671,4.662650,2.0,0.712350,0.712350,0.968533,3.146732


In [201]:
pbp.columns

Index(['play_id', 'game_id', 'old_game_id', 'home_team', 'away_team',
       'season_type', 'week', 'posteam', 'posteam_type', 'defteam',
       ...
       'out_of_bounds', 'home_opening_kickoff', 'qb_epa', 'xyac_epa',
       'xyac_mean_yardage', 'xyac_median_yardage', 'xyac_success', 'xyac_fd',
       'xpass', 'pass_oe'],
      dtype='object', length=372)

#### Creating the Subset

There are 372 variables in the imported play-by-play dataframe. For our analysis, we have determined that we will only need the following:
##### Game and Play Context
- 'game_id': game identifier (string)
- 'play_id': play identifier; unique for a single play (when used along with game_id and drive) (numeric)
- 'home_team': abbreviation; represents the home team in a given game (string)
- 'away_team': abbreviation; represents the away team in a given game (string)
- 'season_type': denotes whether a game occured within the regular season or post season (string)
- 'week': indicates the week of the season in which the game occurred (numeric)
- 'season': indicating the season in which the game was played (numeric)
- 'stadium': location of the game (string)
- 'posteam': indicates the team with possession of the ball during a given play (offense) (string)
- 'defteam': indicates the team on defense (string)
- 'yardline_100': distance (in number of yards) from opponent's end zone (numeric)
- 'game_seconds_remaining': number of seconds remaining in the game (numeric)
- 'qtr': represents quarter of the game (5 is overtime) (numeric)
- 'down': represents the down for the given play (numeric)
- 'ydstogo': indicates distance (in yards) from either the first down marker or the endzone in goal down situations (numeric)
- 'play_type': indicates the type of play: pass (includes sacks), run (includes scrambles), punt, field_goal, kickoff, extra_point, qb_kneel, qb_spike, no_play (timeouts and penalties) (string)
- 'play_clock': time on the playclock when the ball was snapped (string)
- 'posteam_score': represents the score of the team in possession at the start of the play (numeric)
- 'defteam_score': represents the score of the defending team at the start of the play (numeric)
##### Play Results and Penalties
- 'posteam_score_post': represents the score of the team in possession at the end of the play (numeric)
- 'defteam_score_post': represents the score of the defending team at the end of the play (numeric)
- 'epa': represents the expected points added (EPA) by the team in possession for the given play (numeric)
- 'wp': estimated win probabiity for the team in possession given the current situation at the start of the given play (numeric)
- 'wpa': win probability added (WPA) for the team in possession (after the play) (numeric)
- 'yards_gained': yards gained (or lost) by the possessing team, excluding yards gained via fumble recoveries and laterals (numeric)
- 'penalty': binary indicator of whether or not a penalty occurred (numeric)
- 'penalty_team': indicates which team committed the penalty (string)
- 'penalty_yards': yards gained (or lost) by the team in possession from the penalty (numeric)
- 'penalty_type': indicates the penalty type of the first penalty in the given play (string)
- 'replay_or_challenge': binary indicator of whether or not a replay or challenge occurred on the play (numeric)
- 'replay_or_challenge_result': the result of the replay/challenge (string)

In [200]:
# Creating pbp subset

keep_cols_pbp = [
    'game_id', 'play_id', 'home_team', 'away_team', 'season_type', 
    'season', 'week', 'stadium', 'posteam', 'defteam', 'ydstogo', 
    'yardline_100', 'game_seconds_remaining', 'qtr', 'down', 'play_type', 
    'score_differential', 'epa', 'wp', 'wpa', 'penalty', 'penalty_type', 
    'penalty_yards', 'penalty_team', 'yards_gained', 'posteam_score', 
    'defteam_score', 'posteam_score_post', 'defteam_score_post', 
    'replay_or_challenge', 'replay_or_challenge_result', 
    'success', 'play_clock'
]

pbp_subset = pbp[keep_cols_pbp]

In [202]:
pbp_subset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 483605 entries, 0 to 483604
Data columns (total 33 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   game_id                     483605 non-null  object 
 1   play_id                     483605 non-null  float64
 2   home_team                   483605 non-null  object 
 3   away_team                   483605 non-null  object 
 4   season_type                 483605 non-null  object 
 5   season                      483605 non-null  int32  
 6   week                        483605 non-null  int32  
 7   stadium                     483605 non-null  object 
 8   posteam                     457974 non-null  object 
 9   defteam                     457974 non-null  object 
 10  ydstogo                     483605 non-null  float64
 11  yardline_100                449818 non-null  float64
 12  game_seconds_remaining      483341 non-null  float64
 13  qtr           

In [203]:
pbp_subset.head()

,game_id,play_id,home_team,away_team,season_type,season,week,stadium,posteam,defteam,...,penalty_team,yards_gained,posteam_score,defteam_score,posteam_score_post,defteam_score_post,replay_or_challenge,replay_or_challenge_result,success,play_clock
0,2015_01_BAL_DEN,1.0,DEN,BAL,REG,2015,1,Empower Field at Mile High,None,None,...,None,NaN,NaN,NaN,NaN,NaN,0.0,None,0.0,0
1,2015_01_BAL_DEN,36.0,DEN,BAL,REG,2015,1,Empower Field at Mile High,BAL,DEN,...,None,0.0,0.0,0.0,0.0,0.0,0.0,None,0.0,0
2,2015_01_BAL_DEN,51.0,DEN,BAL,REG,2015,1,Empower Field at Mile High,BAL,DEN,...,None,3.0,0.0,0.0,0.0,0.0,0.0,None,0.0,0
3,2015_01_BAL_DEN,75.0,DEN,BAL,REG,2015,1,Empower Field at Mile High,BAL,DEN,...,None,2.0,0.0,0.0,0.0,0.0,0.0,None,0.0,0
4,2015_01_BAL_DEN,96.0,DEN,BAL,REG,2015,1,Empower Field at Mile High,BAL,DEN,...,None,10.0,0.0,0.0,0.0,0.0,0.0,None,1.0,0


### Game Data (2015-2024)
Game data (obtained by 'nfl.load_schedules()') provides game context and information, including the result, betting lines, and, most importantly, the lead official (referee). This will be merged with pbp_subset to obtain a comprehensive dataset for analysis.

In [127]:
# Loading game contextual data
game_data = nfl.load_schedules([2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024])

In [128]:
# Converting to pandas
game_data = game_data.to_pandas()

In [129]:
game_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2743 entries, 0 to 2742
Data columns (total 46 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   game_id           2743 non-null   object 
 1   season            2743 non-null   int32  
 2   game_type         2743 non-null   object 
 3   week              2743 non-null   int32  
 4   gameday           2743 non-null   object 
 5   weekday           2743 non-null   object 
 6   gametime          2743 non-null   object 
 7   away_team         2743 non-null   object 
 8   away_score        2743 non-null   int32  
 9   home_team         2743 non-null   object 
 10  home_score        2743 non-null   int32  
 11  location          2743 non-null   object 
 12  result            2743 non-null   int32  
 13  total             2743 non-null   int32  
 14  overtime          2743 non-null   int32  
 15  old_game_id       2743 non-null   object 
 16  gsis              2743 non-null   int32  


In [130]:
game_data.head()

,game_id,season,game_type,week,gameday,weekday,gametime,away_team,away_score,home_team,...,wind,away_qb_id,home_qb_id,away_qb_name,home_qb_name,away_coach,home_coach,referee,stadium_id,stadium
0,2015_01_PIT_NE,2015,REG,1,2015-09-10,Thursday,20:30,PIT,21,NE,...,7.0,00-0022924,00-0019596,Ben Roethlisberger,Tom Brady,Mike Tomlin,Bill Belichick,Carl Cheffers,BOS00,Gillette Stadium
1,2015_01_IND_BUF,2015,REG,1,2015-09-13,Sunday,13:00,IND,14,BUF,...,15.0,00-0029668,00-0028118,Andrew Luck,Tyrod Taylor,Chuck Pagano,Rex Ryan,John Parry,BUF00,Ralph Wilson Stadium
2,2015_01_GB_CHI,2015,REG,1,2015-09-13,Sunday,13:00,GB,31,CHI,...,11.0,00-0023459,00-0024226,Aaron Rodgers,Jay Cutler,Mike McCarthy,John Fox,Craig Wrolstad,CHI98,Soldier Field
3,2015_01_KC_HOU,2015,REG,1,2015-09-13,Sunday,13:00,KC,27,HOU,...,NaN,00-0023436,00-0026625,Alex Smith,Brian Hoyer,Andy Reid,Bill O'Brien,Peter Morelli,HOU00,NRG Stadium
4,2015_01_CAR_JAX,2015,REG,1,2015-09-13,Sunday,13:00,CAR,20,JAX,...,7.0,00-0027939,00-0031407,Cam Newton,Blake Bortles,Ron Rivera,Gus Bradley,Ron Torbert,JAX00,EverBank Field


#### Creating the Subset
There are a few unneeded/duplicate variables in game_data, so we will only keep the following:
- 'game_id': this is the key variable we will use to merge datasets.
- 'game_type': this is similar to 'season_type' in pbp, but also specifially denotes the type of post season game (WC, DIV, CON, SB) (string)
- 'result': the number of points the home team scored minus the number of points the visiting team scored (numeric)
- 'home_score': the number of points the home team scored (numeric)
- 'away_score': the number of points the away team scored (numeric)
- 'home_moneyline': odds for home team to win the game (numeric)
- 'away_moneyline': odds for away team to win the game (numeric)
- 'spread_line': the spread line for the game: a positive number means the home team was favored by that many points, a negative number means the away team was favored by that many points (lines up with 'result' column) (numeric)
- 'referee': name of the game's referee (head official) (string)

In [224]:
# Creating game_data subset
keep_cols_games = [
    'game_id', 'game_type', 'result', 'home_score', 'away_score', 
    'home_moneyline', 'away_moneyline', 'spread_line', 'referee'
]

game_data_subset = game_data[keep_cols_games]

In [225]:
game_data_subset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2743 entries, 0 to 2742
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   game_id         2743 non-null   object 
 1   game_type       2743 non-null   object 
 2   result          2743 non-null   int32  
 3   home_score      2743 non-null   int32  
 4   away_score      2743 non-null   int32  
 5   home_moneyline  2742 non-null   float64
 6   away_moneyline  2742 non-null   float64
 7   spread_line     2743 non-null   float64
 8   referee         2742 non-null   object 
dtypes: float64(3), int32(3), object(3)
memory usage: 160.9+ KB


In [226]:
game_data_subset.head()

,game_id,game_type,result,home_score,away_score,home_moneyline,away_moneyline,spread_line,referee
0,2015_01_PIT_NE,REG,7,28,21,-350.0,305.0,7.5,Carl Cheffers
1,2015_01_IND_BUF,REG,13,27,14,-101.0,-109.0,-1.0,John Parry
2,2015_01_GB_CHI,REG,-8,23,31,222.0,-250.0,-5.5,Craig Wrolstad
3,2015_01_KC_HOU,REG,-7,20,27,-105.0,-105.0,-1.0,Peter Morelli
4,2015_01_CAR_JAX,REG,-11,9,20,131.0,-145.0,-3.0,Ron Torbert


## Merging
(There were no critical nulls (none in game_id), so merge then clean)

In [227]:
# Merging pbp_subset and game_data_subset
plays_and_games = pbp_subset.merge(
    game_data_subset,
    on = 'game_id',
    how = 'left'
)

In [228]:
plays_and_games.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 483605 entries, 0 to 483604
Data columns (total 41 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   game_id                     483605 non-null  object 
 1   play_id                     483605 non-null  float64
 2   home_team                   483605 non-null  object 
 3   away_team                   483605 non-null  object 
 4   season_type                 483605 non-null  object 
 5   season                      483605 non-null  int32  
 6   week                        483605 non-null  int32  
 7   stadium                     483605 non-null  object 
 8   posteam                     457974 non-null  object 
 9   defteam                     457974 non-null  object 
 10  ydstogo                     483605 non-null  float64
 11  yardline_100                449818 non-null  float64
 12  game_seconds_remaining      483341 non-null  float64
 13  qtr           

In [229]:
plays_and_games.head()

,game_id,play_id,home_team,away_team,season_type,season,week,stadium,posteam,defteam,...,success,play_clock,game_type,result,home_score,away_score,home_moneyline,away_moneyline,spread_line,referee
0,2015_01_BAL_DEN,1.0,DEN,BAL,REG,2015,1,Empower Field at Mile High,None,None,...,0.0,0,REG,6,19,13,-220.0,197.0,4.5,Gene Steratore
1,2015_01_BAL_DEN,36.0,DEN,BAL,REG,2015,1,Empower Field at Mile High,BAL,DEN,...,0.0,0,REG,6,19,13,-220.0,197.0,4.5,Gene Steratore
2,2015_01_BAL_DEN,51.0,DEN,BAL,REG,2015,1,Empower Field at Mile High,BAL,DEN,...,0.0,0,REG,6,19,13,-220.0,197.0,4.5,Gene Steratore
3,2015_01_BAL_DEN,75.0,DEN,BAL,REG,2015,1,Empower Field at Mile High,BAL,DEN,...,0.0,0,REG,6,19,13,-220.0,197.0,4.5,Gene Steratore
4,2015_01_BAL_DEN,96.0,DEN,BAL,REG,2015,1,Empower Field at Mile High,BAL,DEN,...,1.0,0,REG,6,19,13,-220.0,197.0,4.5,Gene Steratore


## Handling NAs
To clean the data, we will first check where nulls are present in the info table. The first category to fix are:
- 'penalty_yards', 'penalty_type', 'penalty_team': null when 'penalty' = 0
- 'replay_or_challenge_result': null when 'replay_or_challenge' = 0

These null values are legitimate parts of data, and are to be expected (if a penalty/challenge didn't happen, it makes sense for penalty/challenge-related qualifiers to be null). So, we will set all null values of 'penalty_yards' to be '0' (since it's a numeric variable), and null values of 'penalty_type', 'penalty_team', and 'replay_or_challenge_result' to be 'None' (since these are character variables). 

In [234]:
# Setting NAs to 0 for penalty_yards; 'None' for penalty_type, penalty_team, replay...result
plays_and_games['penalty_type'] = plays_and_games['penalty_type'].fillna('None')
plays_and_games['penalty_team'] = plays_and_games['penalty_team'].fillna('None')
plays_and_games['penalty_yards'] = plays_and_games['penalty_yards'].fillna(0)
plays_and_games['replay_or_challenge_result'] = plays_and_games['replay_or_challenge_result'].fillna('None')

Now, we will drop all null values that don't represent a play. For example, null values of 'down' represent kickoffs, PATs, some penalties, and some special plays, while 'play_type' wouldn't have a reason for this besides being a malformed entry (these account for small percentages of the total entries). For some variables, like 'posteam' and 'defteam', we will first verify whether they belong in the former category by checking to see how the data classifies the possession team and defending team on kickoffs, as kicking and returning teams on these plays aren't always sorted this way. 

In [245]:
# verifying null posteam/defteam isn't due to kickoffs
plays_and_games[plays_and_games['play_type']== 'kickoff'].info()

<class 'pandas.core.frame.DataFrame'>
Index: 27945 entries, 1 to 483600
Data columns (total 41 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   game_id                     27945 non-null  object 
 1   play_id                     27945 non-null  float64
 2   home_team                   27945 non-null  object 
 3   away_team                   27945 non-null  object 
 4   season_type                 27945 non-null  object 
 5   season                      27945 non-null  int32  
 6   week                        27945 non-null  int32  
 7   stadium                     27945 non-null  object 
 8   posteam                     27945 non-null  object 
 9   defteam                     27945 non-null  object 
 10  ydstogo                     27945 non-null  float64
 11  yardline_100                27945 non-null  float64
 12  game_seconds_remaining      27945 non-null  float64
 13  qtr                         27945 n

This demonstrates that the nulls found in 'posteam' and 'defteam' aren't due to kickoffs, so they can be removed (side note: this also demonstrates how 'down' is NA on kickoff plays --- if there were any non-nulls here, we would have a problem).

In [231]:
# Dropping NAs for posteam and defteam
plays_and_games = plays_and_games.dropna(subset=['posteam','defteam'])

We will also drop NAs for 'referee', as this variable is needed for our analysis:

In [232]:
# Dropping NAs for referee
plays_and_games = plays_and_games.dropna(subset=['referee'])

We can now move forward with dropping remaining null values in the following variables, as NAs indicate missing data that could affect our analysis:

In [243]:
# dropping rows where NAs are present (for consistency)
# rows where play_type is NA
plays_and_games = plays_and_games.dropna(subset=['play_type'])

In [236]:
# NAs in yards_gained
plays_and_games = plays_and_games.dropna(subset=['yards_gained'])

In [237]:
# NAs in epa
plays_and_games = plays_and_games.dropna(subset=['epa'])

In [238]:
# NAs in wpa
plays_and_games = plays_and_games.dropna(subset=['wpa'])

In [239]:
# NAs in yardline_100
plays_and_games = plays_and_games.dropna(subset=['yardline_100'])

In [241]:
# NAs in home_moneyline, away_moneyline
plays_and_games = plays_and_games.dropna(subset=['home_moneyline', 'away_moneyline'])

In [244]:
plays_and_games.info()

<class 'pandas.core.frame.DataFrame'>
Index: 444258 entries, 1 to 483600
Data columns (total 41 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   game_id                     444258 non-null  object 
 1   play_id                     444258 non-null  float64
 2   home_team                   444258 non-null  object 
 3   away_team                   444258 non-null  object 
 4   season_type                 444258 non-null  object 
 5   season                      444258 non-null  int32  
 6   week                        444258 non-null  int32  
 7   stadium                     444258 non-null  object 
 8   posteam                     444258 non-null  object 
 9   defteam                     444258 non-null  object 
 10  ydstogo                     444258 non-null  float64
 11  yardline_100                444258 non-null  float64
 12  game_seconds_remaining      444258 non-null  float64
 13  qtr                

## Filtering the Subset

Now that the dataframe only contains the variables we care about for this analysis, we can filter it to only include plays in which the officiating crew had involvement. This will be a separate dataset ('play_off') that can be used for tracking pure frequencies of calls.

In [246]:
# Filtering the subset
play_off = plays_and_games[
    (plays_and_games['penalty'] == 1) | 
    (plays_and_games['replay_or_challenge'] == 1)
].copy()

In [247]:
play_off.info()

<class 'pandas.core.frame.DataFrame'>
Index: 37711 entries, 14 to 483591
Data columns (total 41 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   game_id                     37711 non-null  object 
 1   play_id                     37711 non-null  float64
 2   home_team                   37711 non-null  object 
 3   away_team                   37711 non-null  object 
 4   season_type                 37711 non-null  object 
 5   season                      37711 non-null  int32  
 6   week                        37711 non-null  int32  
 7   stadium                     37711 non-null  object 
 8   posteam                     37711 non-null  object 
 9   defteam                     37711 non-null  object 
 10  ydstogo                     37711 non-null  float64
 11  yardline_100                37711 non-null  float64
 12  game_seconds_remaining      37711 non-null  float64
 13  qtr                         37711 

## Other Data from nflreadpy
nflreadpy also allows us to import secondary datasets that will be useful in our final report. First, we will load team data, which includes team IDs, logos, and color codes. This will aid in our final visualizations. 

In [248]:
# Team colors, logos, and IDs for visualizations
team_attributes = nfl.load_teams()

In [249]:
# Converting to pandas
team_attributes = team_attributes.to_pandas()

In [250]:
team_attributes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36 entries, 0 to 35
Data columns (total 16 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   team_abbr             36 non-null     object
 1   team_name             36 non-null     object
 2   team_id               36 non-null     int32 
 3   team_nick             36 non-null     object
 4   team_conf             36 non-null     object
 5   team_division         36 non-null     object
 6   team_color            36 non-null     object
 7   team_color2           36 non-null     object
 8   team_color3           35 non-null     object
 9   team_color4           35 non-null     object
 10  team_logo_wikipedia   36 non-null     object
 11  team_logo_espn        36 non-null     object
 12  team_wordmark         36 non-null     object
 13  team_conference_logo  36 non-null     object
 14  team_league_logo      36 non-null     object
 15  team_logo_squared     36 non-null     obje

In [251]:
team_attributes.head()

,team_abbr,team_name,team_id,team_nick,team_conf,team_division,team_color,team_color2,team_color3,team_color4,team_logo_wikipedia,team_logo_espn,team_wordmark,team_conference_logo,team_league_logo,team_logo_squared
0,ARI,Arizona Cardinals,3800,Cardinals,NFC,NFC West,#97233F,#000000,#ffb612,#a5acaf,https://upload.wikimedia.org/wikipedia/en/thum...,https://a.espncdn.com/i/teamlogos/nfl/500/ari.png,https://github.com/nflverse/nflverse-pbp/raw/m...,https://github.com/nflverse/nflverse-pbp/raw/m...,https://raw.githubusercontent.com/nflverse/nfl...,https://github.com/nflverse/nflverse-pbp/raw/m...
1,ATL,Atlanta Falcons,200,Falcons,NFC,NFC South,#A71930,#000000,#a5acaf,#a30d2d,https://upload.wikimedia.org/wikipedia/en/thum...,https://a.espncdn.com/i/teamlogos/nfl/500/atl.png,https://github.com/nflverse/nflverse-pbp/raw/m...,https://github.com/nflverse/nflverse-pbp/raw/m...,https://raw.githubusercontent.com/nflverse/nfl...,https://github.com/nflverse/nflverse-pbp/raw/m...
2,BAL,Baltimore Ravens,325,Ravens,AFC,AFC North,#241773,#9E7C0C,#9e7c0c,#c60c30,https://upload.wikimedia.org/wikipedia/en/thum...,https://a.espncdn.com/i/teamlogos/nfl/500/bal.png,https://github.com/nflverse/nflverse-pbp/raw/m...,https://github.com/nflverse/nflverse-pbp/raw/m...,https://raw.githubusercontent.com/nflverse/nfl...,https://github.com/nflverse/nflverse-pbp/raw/m...
3,BUF,Buffalo Bills,610,Bills,AFC,AFC East,#00338D,#C60C30,#0c2e82,#d50a0a,https://upload.wikimedia.org/wikipedia/en/thum...,https://a.espncdn.com/i/teamlogos/nfl/500/buf.png,https://github.com/nflverse/nflverse-pbp/raw/m...,https://github.com/nflverse/nflverse-pbp/raw/m...,https://raw.githubusercontent.com/nflverse/nfl...,https://github.com/nflverse/nflverse-pbp/raw/m...
4,CAR,Carolina Panthers,750,Panthers,NFC,NFC South,#0085CA,#000000,#bfc0bf,#0085ca,https://upload.wikimedia.org/wikipedia/en/thum...,https://a.espncdn.com/i/teamlogos/nfl/500-dark...,https://github.com/nflverse/nflverse-pbp/raw/m...,https://github.com/nflverse/nflverse-pbp/raw/m...,https://raw.githubusercontent.com/nflverse/nfl...,https://github.com/nflverse/nflverse-pbp/raw/m...


### Officials Data
The officials dataset will also be used to reference specific members of a given officiating crew assigned to each game, which includes Referee, Umpire, Down Judge, Line Judge, Field Judge, Side Judge, and Back Judge. 

In [252]:
# Loading officiating data
officials = nfl.load_officials([2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024])

In [253]:
# Converting to pandas
officials = officials.to_pandas()

In [254]:
officials.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19834 entries, 0 to 19833
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   game_id        19834 non-null  object
 1   game_key       19834 non-null  object
 2   official_name  19834 non-null  object
 3   position       19834 non-null  object
 4   jersey_number  19834 non-null  int32 
 5   official_id    19834 non-null  object
 6   season         19834 non-null  int32 
 7   season_type    19834 non-null  object
 8   week           19834 non-null  int32 
dtypes: int32(3), object(6)
memory usage: 1.1+ MB


In [256]:
officials.head()

,game_id,game_key,official_name,position,jersey_number,official_id,season,season_type,week
0,2015091000,56503,Brad Freeman,Field Judge,88,25,2015,REG,1
1,2015091000,56503,Kent Payne,Head Linesman,79,28,2015,REG,1
2,2015091000,56503,Terrence Miles,Back Judge,111,139,2015,REG,1
3,2015091000,56503,Tim Podraza,Line Judge,47,123,2015,REG,1
4,2015091000,56503,Scott Novak,Side Judge,1,94,2015,REG,1


## Saving to csv Files
For the sake of consistency, we will save these datasets to csv files:

In [257]:
# plays_and_games
plays_and_games.to_csv("plays_and_games.csv", index=False)

In [258]:
# play_off
play_off.to_csv("play_off.csv", index=False)

In [259]:
# team_attributes
team_attributes.to_csv("team_attributes.csv", index=False)

In [260]:
# officials
officials.to_csv("officials.csv", index=False)